# 🚀 SimpleAI — TinyGPT 加法机制探针 (Google Colab 极速运行手册)

本 Notebook 提供了在 Google Colab 上使用 GPU/CPU 训练与评测 **additive-rand-transformer** 的完整流水线。

### 核心特性：
1. **零额外配置直连 Hugging Face**：Colab 位于海外网络，直连 Hugging Face 极速下载代码与权重（公开仓库免 Token，私有仓库支持一键填入）。
2. **Google Drive 深度集成**：自动挂载并双向同步代码、配置与训练 Checkpoint (`.pt`)。
3. **JSON 配置灵活拉起**：支持通过 `train.py --config config.json` 任意指定架构、超参和数据源。
4. **学术机制诊断与 INT8 量化**：H1 草稿纸探针、动态量化无损压缩比评测、交互式 REPL。

## 步骤 1：挂载 Google Drive 并配置工作区

In [ ]:
from google.colab import drive
import os, sys

# 1. 挂载 Google Drive（可选，若不需要存入 Drive 也可跳过此步）
try:
    drive.mount('/content/drive')
    DRIVE_WORKSPACE = '/content/drive/MyDrive/simpleAI_workspace'
    os.makedirs(f'{DRIVE_WORKSPACE}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_WORKSPACE}/runs', exist_ok=True)
    print(f'✓ Google Drive 工作目录就绪: {DRIVE_WORKSPACE}')
except Exception as e:
    print(f'Drive 挂载提示: {e}，将使用 Colab 本地临时存储。')
    DRIVE_WORKSPACE = None

## 步骤 2：环境准备与 Hugging Face 代码下载
Colab 海外环境直连 `huggingface.co`，速度通常可达 100MB/s+，**公开仓库完全不需要任何 Token**。

In [ ]:
# 1. 安装核心依赖
!pip install -q torch openpyxl huggingface_hub pandas matplotlib

# 2. 从 Hugging Face 克隆或下载代码
%cd /content
if not os.path.exists('/content/additive-rand-transformer'):
    print('正在从 Hugging Face 克隆仓库...')
    !git clone https://huggingface.co/Hana-ame/additive-rand-transformer /content/additive-rand-transformer

%cd /content/additive-rand-transformer
if '/content/additive-rand-transformer' not in sys.path:
    sys.path.insert(0, '/content/additive-rand-transformer')
print('✓ 代码仓库与 Python 路径已就绪')

## 步骤 3：从 Hugging Face 极速下载预训练权重 (.pt Checkpoints)
> **公开仓库免配置**；若为私有仓库，可在左侧 Secrets（🔑钥匙图标）中添加 `HF_TOKEN` 或在下方填入 token。

In [ ]:
import os, torch
from huggingface_hub import hf_hub_download

# 如果是私有仓库，尝试从 Colab Secrets 获取 token，公开仓库则为 None
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', None)

REPO_ID = 'Hana-ame/additive-rand-transformer'
CKPT_NAME = 'l4_d128_cot_bias05_final.pt'
os.makedirs('checkpoints', exist_ok=True)
local_path = f'checkpoints/{CKPT_NAME}'

print(f'正在从 Hugging Face 仓库 ({REPO_ID}) 下载权重: {CKPT_NAME} ...')
try:
    downloaded_file = hf_hub_download(
        repo_id=REPO_ID,
        filename=f'checkpoints/{CKPT_NAME}',
        local_dir='.',
        token=hf_token
    )
    size_mb = os.path.getsize(local_path) / (1024 * 1024)
    print(f'✓ 权重下载成功: {local_path} ({size_mb:.2f} MB)')
    
    # 自动备份至 Google Drive
    if DRIVE_WORKSPACE:
        !cp {local_path} {DRIVE_WORKSPACE}/checkpoints/{CKPT_NAME}
        print(f'✓ 已同步备份到 Google Drive')
except Exception as e:
    print(f'下载提示: {e}')

## 步骤 4：生成自定义实验配置 (`config.json`)

In [ ]:
import json

custom_config = {
    'layers': 4,                  # 模型层数 L (1-10)
    'd': 128,                     # 嵌入维度 d (32-512)
    'heads': 4,                   # 注意力头数
    'steps': 4000,                # 训练步数 (建议 2000-4000)
    'batch_size': 32,             # 批量大小
    'lr': 3e-4,                   # 学习率
    'wd': 0.1,                    # 权重衰减
    'warmup': 200,                # 预热步数
    'datasource': {
        'type': 'cot',            # cot: 竖式草稿纸 | plain: 无中间过程
        'max_digits': 4,          # 最大操作数位数 (1-4位)
        'bias': 0.5,              # 4位高难度进位加权 (0.5 为最佳相变点)
        'max_spaces': 3,          # 运算符两侧空格随机扰动
        'single': True            # 单样本训练（不跨题打包）
    }
}

with open('config.json', 'w', encoding='utf-8') as f:
    json.dump(custom_config, f, indent=2)

print('✓ 已生成 config.json:')
print(json.dumps(custom_config, indent=2))

## 步骤 5：启动训练 (`train.py --config config.json`)

In [ ]:
# 执行训练并输出实时进度与 CoT 准确率
!python -m additive_rand_transformer.train --config config.json

## 步骤 6：机制探针学术诊断 (H1 草稿纸篡改测试)

In [ ]:
# 运行 H1 草稿纸篡改敏感度探针
!python -m additive_rand_transformer.explore_h1 || true

## 步骤 7：模型 INT8 动态量化评测 (压缩比与精度验证)

In [ ]:
# 运行 PyTorch Dynamic INT8 量化基准
!python -m additive_rand_transformer.quantize --checkpoint checkpoints/l4_d128_cot_bias05_final.pt

## 步骤 8：交互式推理体验 (REPL)

In [ ]:
from additive_rand_transformer.model import TinyGPT, TinyGPTConfig
from additive_rand_transformer.data import BOS, EOS, SP, PLUS, MINUS, _int_to_tokens, decode

ckpt_file = 'checkpoints/l4_d128_cot_bias05_final.pt'
if not os.path.exists(ckpt_file):
    import glob
    ckpts = sorted(glob.glob('runs/**/checkpoint*.pt', recursive=True))
    if ckpts:
        ckpt_file = ckpts[-1]

if os.path.exists(ckpt_file):
    ck = torch.load(ckpt_file, map_location='cpu', weights_only=False)
    cfg = TinyGPTConfig(**{k: v for k, v in ck['config'].items() if k in TinyGPTConfig.__dataclass_fields__})
    model = TinyGPT(cfg)
    model.load_state_dict(ck['model'])
    model.eval()
    print(f'✓ 成功加载模型: {ckpt_file} (L={cfg.n_layer}, d={cfg.n_embd}, {model.num_parameters():,} 参数)')

    def calculate(a, b, op='+'):
        op_id = PLUS if op == '+' else MINUS
        prompt = [BOS] + _int_to_tokens(a) + [SP, op_id, SP] + _int_to_tokens(b) + [SP]
        ids = list(prompt)
        with torch.no_grad():
            for _ in range(80):
                x = torch.tensor([ids], dtype=torch.long)
                logits, _ = model(x, None)
                nxt = int(logits[0, -1].argmax())
                ids.append(nxt)
                if nxt == EOS:
                    break
        result_str = decode(ids)
        print(f'【题目】: {a} {op} {b}')
        print(f'【模型生成 (含竖式CoT)】:\n{result_str}\n')

    # 演示多位计算
    calculate(37, 85, '+')
    calculate(523, 194, '-')
    calculate(1234, 5678, '+')
    calculate(9999, 4321, '-')
else:
    print('未找到 checkpoint，请先运行步骤 3 或步骤 5。')

## 步骤 9：将全部产物备份至 Google Drive

In [ ]:
if DRIVE_WORKSPACE:
    !cp -ru runs/ {DRIVE_WORKSPACE}/runs/ || true
    !cp -ru checkpoints/ {DRIVE_WORKSPACE}/checkpoints/ || true
    !cp config.json {DRIVE_WORKSPACE}/ || true
    print(f'✓ 全部权重、配置与运行日志已成功归档至 Google Drive: {DRIVE_WORKSPACE}')
else:
    print('未挂载 Google Drive，产物保存在 Colab 本地。')